# Unparallel idler and nip loading

Run all cells to reproduce five figures: four winding-line results from one FMU and a friction comparison calculated below. The FMU is exported from **Roll2RollDynamics**, an internal Modelica library for web handling; the library itself is not included. Two further figures, the nip-load and nip-friction sweeps, come from `python notebook_utils.py load-sweep` and `python notebook_utils.py friction-sweep`.

The FMU contains Windows 64-bit binaries. Install `requirements.txt` in this notebook's Python environment. The winding-line simulations use Model Exchange with CVode.

The single archive takes `idlerYaw`, `idlerTram`, `nipRunout` and `nipCompression` before initialization. The idler is mounted through a run-time rotation and the nip drum through a run-time runout, so one export covers the aligned idler and every fault below.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Image, Markdown, display

study_dir = Path.cwd()
if not (study_dir / "notebook_utils.py").is_file():
    raise FileNotFoundError("Open this notebook from its own folder.")
sys.path.insert(0, str(study_dir))

from notebook_utils import (
    BUILD_DIR,
    CASES,
    plot_friction_result,
    plot_unparallel_results,
    run_cases,
    summary_table,
)

force_simulation = False  # True reruns every FMU case; otherwise matching caches are reused.
cache_dir = BUILD_DIR / "cache"
figure_dir = BUILD_DIR / "figures"


def show(paths):
    for path in paths:
        display(Image(filename=str(path)))


## Yaw, tram and runout sweeps with the bearing-only baseline

Nine cases, 40 s each: the nip clear at 1.15 deg yaw, the loaded machine-axis nip at 0, 1.15, 2 and 3 deg of yaw, the same nip at 1.15, 2 and 3 deg of tram, and on the aligned idler a nip drum with 0.3 mm radial runout. The table lists the crossing and wedge angles the contact sees, the edge lift and end opening, the touching width of the face and where the load resultant acts, the nip load band at speed, the slip and tracking results, and the peak-to-peak tension ripple of the spans entering and leaving the idler.

Nip force integrates linear cover compression across the contacting face. The 1 mm aligned compression gives 1000 N; the external position constraint can produce forces above the maximum actuator force parameter.


In [ ]:
line_runs = run_cases(list(CASES), cache_dir=cache_dir, force=force_simulation)
line_figures = {
    path.stem: path
    for path in plot_unparallel_results(line_runs, figure_dir / "MisalignedIdler")
}
display(Markdown(summary_table(line_runs)))


In [ ]:
show([line_figures["bearing-vs-nip"]])


## What the yaw does to the nip, and what the tram does to the web

The first figure follows the yaw: the part of the nip face in contact with its steady load, idler slip through the startup and the web walk. The second follows the tram: lateral walk, the one-sided footprint with the load centre, and accumulated idler slip against the aligned case.


In [ ]:
show([line_figures["yaw-sweep"], line_figures["tram-sweep"]])


## What runout does that misalignment cannot

A fixed yaw or tram gives a fixed gap across the face and a steady load. A drum that is eccentric on its bearing axis changes the gap once per nip turn. The figure shows half a second at speed: the nip load, and the tensions of the spans entering and leaving the idler relative to the aligned case.


In [ ]:
show([line_figures["runout"]])


## Stribeck and Triple S

The following cell evaluates the same three cubic transitions used by the Modelica `tripleS` function, alongside a signed Stribeck curve. This is an algebraic calculation and needs no FMU. Both curves use the study's dry-friction parameters: adhesion peak 0.35 at 1 mm/s, sliding coefficient 0.22 at 3 mm/s, and zero viscous slope.

Triple S permits small creep near adhesion and passes continuously through zero. The signed Stribeck curve has a jump at zero, which is left open in the plot.


In [ ]:
def s_curve(x, x_min, x_max, y_min, y_max):
    """Evaluate the clamped cubic transition used by Modelica sFunction."""
    scaled = np.clip(2 * (x - (x_min + x_max) / 2) / (x_max - x_min), -1, 1)
    return (-0.5 * scaled**3 + 1.5 * scaled) * (y_max - y_min) / 2 + (y_max + y_min) / 2


v_adhesion, v_slide = 0.001, 0.003
mu_adhesion, mu_sliding = 0.35, 0.22
viscous_slope = 0.0
velocity = np.linspace(-0.02, 0.02, 2001)
triple_s = np.where(
    velocity > v_adhesion,
    s_curve(velocity, v_adhesion, v_slide, mu_adhesion, mu_sliding),
    np.where(
        velocity < -v_adhesion,
        s_curve(velocity, -v_slide, -v_adhesion, -mu_sliding, -mu_adhesion),
        s_curve(velocity, -v_adhesion, v_adhesion, -mu_adhesion, mu_adhesion),
    ),
) + viscous_slope * velocity
stribeck = np.sign(velocity) * (
    mu_sliding + (mu_adhesion - mu_sliding) * np.exp(-(velocity / v_adhesion)**2)
) + viscous_slope * velocity
friction_run = {
    "slipVelocity": velocity,
    "tripleSCoefficient": triple_s,
    "stribeckCoefficient": stribeck,
}
friction_figure = plot_friction_result(friction_run, figure_dir / "MisalignedIdler")
show([friction_figure])


## Saved figures

All plot axes are linear. Simulation caches are keyed by FMU contents and run settings; the friction curves are recalculated on every run. Figures and caches are saved under `build/notebook-fmu` beside this notebook.


In [ ]:
all_figures = list(line_figures.values()) + [friction_figure]
assert len(all_figures) == 5
for path in all_figures:
    print(path.relative_to(BUILD_DIR))
